# Exploração da API SGS do Banco Central

Objetivo: entender o formato de resposta da API do BCB, com foco inicial na
série 432 (Taxa Selic).

**Referência:** https://dadosabertos.bcb.gov.br/dataset/sgs

In [3]:
import requests
import json
from datetime import datetime

# URL base do SGS
URL_BASE = "https://api.bcb.gov.br/dados/serie/bcdata.sgs."

print("✅ Imports OK")
print(f"URL base: {URL_BASE}")

✅ Imports OK
URL base: https://api.bcb.gov.br/dados/serie/bcdata.sgs.


In [4]:
# Pegar os últimos 5 valores da Selic (série 432)
url = f"{URL_BASE}432/dados/ultimos/5?formato=json"

print(f"🔎 Consultando: {url}\n")

resposta = requests.get(url, timeout=10)
print(f"Status HTTP: {resposta.status_code}")
print(f"Content-Type: {resposta.headers.get('Content-Type')}")

🔎 Consultando: https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados/ultimos/5?formato=json

Status HTTP: 200
Content-Type: application/json; charset=utf-8


In [5]:
# Ver o texto da resposta (antes de transformar em JSON)
print("Conteúdo cru:")
print(resposta.text)

Conteúdo cru:
[{"data":"31/10/2026","valor":"13.75"},{"data":"01/11/2026","valor":"13.75"},{"data":"02/11/2026","valor":"13.75"},{"data":"03/11/2026","valor":"13.75"},{"data":"04/11/2026","valor":"13.75"}]


In [6]:
# Transformar em estrutura Python
dados = resposta.json()

print(f"Tipo do objeto: {type(dados)}")
print(f"Quantidade de itens: {len(dados)}")
print()
print("Primeiro item:")
print(dados[0])
print()
print("Último item:")
print(dados[-1])

Tipo do objeto: <class 'list'>
Quantidade de itens: 5

Primeiro item:
{'data': '31/10/2026', 'valor': '13.75'}

Último item:
{'data': '04/11/2026', 'valor': '13.75'}


In [7]:
# Examinar cada item individualmente
for item in dados:
    data = item["data"]
    valor = item["valor"]
    print(f"  {data}  →  {valor}")

  31/10/2026  →  13.75
  01/11/2026  →  13.75
  02/11/2026  →  13.75
  03/11/2026  →  13.75
  04/11/2026  →  13.75


In [8]:
# IMPORTANTE: verificar se "valor" é número ou string
primeiro = dados[0]
print(f"Tipo de 'valor' no JSON: {type(primeiro['valor'])}")
print(f"Tipo de 'data' no JSON:  {type(primeiro['data'])}")
print()

# Tentar operação matemática
try:
    soma = float(primeiro["valor"]) + 1
    print(f"✅ Convertido para float: {soma}")
except Exception as e:
    print(f"❌ Erro: {e}")

Tipo de 'valor' no JSON: <class 'str'>
Tipo de 'data' no JSON:  <class 'str'>

✅ Convertido para float: 14.75


In [9]:
# Explorar conversão de data: BCB usa DD/MM/AAAA, queremos ISO 8601 (AAAA-MM-DD)
from datetime import datetime

data_bcb = "04/11/2026"  # formato BCB

# Converter para objeto datetime
dt = datetime.strptime(data_bcb, "%d/%m/%Y")
print(f"Objeto datetime: {dt}")
print(f"Tipo: {type(dt)}")
print()

# Formatar como ISO 8601
data_iso = dt.strftime("%Y-%m-%d")
print(f"Data ISO 8601: {data_iso}")

Objeto datetime: 2026-11-04 00:00:00
Tipo: <class 'datetime.datetime'>

Data ISO 8601: 2026-11-04


## Conclusões da exploração

Após investigar a API SGS do BCB, sabemos que:

1. **URL padrão:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.{CODIGO}/dados/{FILTRO}?formato=json`
2. **Retorno:** array de objetos `{"data": "DD/MM/AAAA", "valor": "string"}`
3. **Valores são strings** — precisam ser convertidos para `float`
4. **Datas são DD/MM/AAAA** — precisam ser convertidas para ISO 8601
5. **Sem autenticação** — a API é pública
6. **Filtros úteis:** `ultimos/N` e `dataInicial`/`dataFinal`

### Próximo passo

Traduzir essa exploração em um **script Python** (`python/scripts/coletores/bcb.py`)
que:
- Faz a chamada HTTP
- Converte datas e valores
- Retorna os dados no formato que o `dados.json` precisa

## Expansão — séries 20622 (Crédito) e 20542 (Endividamento)

In [11]:
# Testar as séries novas
for codigo in [20622, 29037]:
    url = f"{URL_BASE}{codigo}/dados/ultimos/3?formato=json"
    r = requests.get(url, timeout=10)
    print(f"\n--- Série {codigo} ---")
    print(f"Status: {r.status_code}")
    if r.status_code == 200:
        for item in r.json():
            print(f"  {item['data']} → {item['valor']}")
    else:
        print(f"Erro: {r.text[:200]}")


--- Série 20622 ---
Status: 200
  01/07/2026 → 55.60
  01/06/2026 → 55.75
  01/05/2026 → 55.74

--- Série 29037 ---
Status: 200
  01/04/2026 → 49.88
  01/05/2026 → 49.83
  01/06/2026 → 49.75
